In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import os
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import torchvision
import numpy as np


# Define transformations
transform = transforms.Compose([
    transforms.Resize((32, 32)),  # Resize images
    transforms.RandomRotation(15),  # Rotate images randomly within ±15 degrees
    transforms.ToTensor(),  # Convert to tensor
    transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5]),  # value for each channel
])

transform_valid_test = transforms.Compose([
    transforms.Resize((32, 32)),  # Resize images
    transforms.ToTensor(),  # Convert to tensor
    transforms.Normalize(mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5]),
])

# Initialize dataset for Train
train_path = os.path.join(path, "PlantVillage", "train")
test_path = os.path.join(path, "PlantVillage", "test")

# Write your code here
train_dataset = ImageFolder(train_path, transform=transform)
test_dataset = ImageFolder(test_path, transform=transform_valid_test)

classes = ['Potato__Early_blight', 'Potato__Late_blight', 'Potato__healthy']

# Create DataLoader
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True, num_workers=2)


def imshow(img):
    img = img / 2 + 0.5  # unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

# Get some random training images
dataiter = iter(train_loader)
images, labels = next(dataiter)

# Show images
imshow(torchvision.utils.make_grid(images[:4]))
# Print labels
print(' '.join(f'{classes[labels[j]]:5s}' for j in range(4)))



In [ ]:
import torch.nn as nn
import torch

# Write your code here
class CustomModel(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(

            # TRACE THE SHAPE OF THE TENSORS AS IT PASSES THROUGH THE CONV LAYERS TO AVOID SHAPE MISMATCH ERRORS
            # HOW? --> by using the output features formula shown above

            nn.Conv2d(1, 32, kernel_size=3, padding=1),  # [B,32,28,28]
            nn.ReLU(),
            nn.MaxPool2d(2),                             # [B,32,14,14]

            nn.Conv2d(32, 64, kernel_size=3, padding=1), # [B,64,14,14]
            nn.ReLU(),
            nn.MaxPool2d(2),                             # [B,64,7,7]

            nn.Conv2d(32, 64, kernel_size=3, padding=1), # [B,64,14,14]
            nn.ReLU(),
            nn.MaxPool2d(2),                             # [B,64,7,7]

            nn.Conv2d(32, 64, kernel_size=3, padding=1), # [B,64,14,14]
            nn.ReLU(),
            nn.MaxPool2d(2),                             # [B,64,7,7]

            nn.Conv2d(32, 64, kernel_size=3, padding=1), # [B,64,14,14]
            nn.ReLU(),
            nn.MaxPool2d(2),                             # [B,64,7,7]
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),                                # [B, 64*7*7]
            nn.Linear(32 * 3 * 3, 3),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(3, num_classes)                  # logits
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:
# Write your code here
# Training and Validation Loops
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()

def validate(model, dataloader, criterion, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            correct += (outputs.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)
    return 100 * correct / total  # Return accuracy



In [ ]:
import torch.optim as optim

# Write your code here
# Run Training
model = CustomModel()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001)

for epoch in range(5):  # Train for 5 epochs
    train_one_epoch(model, train_loader, criterion, optimizer, device)
    accuracy = validate(model, test_loader, criterion, device)
    print(f"Epoch {epoch+1}: Validation Accuracy = {accuracy:.2f}%")

In [ ]:
# Write your code here
